# Jeff original notebook — recorded source revision

Use a fresh Colab **L4** GPU runtime. This runs the original notebook 83 runner from commit `8ad4afeb347c16baeecfcfd9b7640608ac2f072b`, without modifying its sampler, model loader, rollout or success check. Both historical arms run: stock with measurement-only uncertainty probes, and P&P refinement. Two shards run sequentially: 400 episodes per arm, 800 rollouts total. No Q correction.

Setup pins the recorded Torch/CUDA versions and checks out the historical source instead of the moving main branch. A fresh experiment name prevents reuse or overwrite of Jeff’s reference results. Historical model-file revisions and several environment versions were not recorded; using this original code does not recover those missing details. Results and trajectories use the original Supabase logging. Rerunning resumes the same experiment.

Before rollout, this notebook verifies the 400 state identities, task-definition hashes and episode seeds against Jeff’s saved stock rows. It saves the reference, recovered run metadata, exact source/runtime/grader fingerprints, and the installed dependency list. The final report includes paired changes, uncertainty by episode and by task, and separate comparisons for the historical source revisions. Unknown historical settings remain explicitly unknown.


In [ ]:
import os, sys, subprocess
from pathlib import Path
from google.colab import userdata
for key in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY', 'HF_TOKEN'):
    try:
        value = userdata.get(key)
        if value: os.environ[key] = value
    except Exception:
        pass
for key in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY', 'HF_TOKEN'):
    if not os.environ.get(key):
        raise RuntimeError(f'Add {key} in Colab Secrets and allow notebook access.')
if any(name == 'pnp' or name.startswith('pnp.') for name in sys.modules):
    raise RuntimeError('Start a fresh runtime: another pnp source version is already imported.')
SOURCE_REVISION = '8ad4afeb347c16baeecfcfd9b7640608ac2f072b'
repo_dir = Path('/content/jeff-original-source')
if not (repo_dir / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/ArjunS07/cs159-sp26.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', SOURCE_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_REVISION
assert not subprocess.check_output(['git', '-C', str(repo_dir), 'status', '--porcelain'], text=True).strip()
package_dir = repo_dir / 'pnp-vla'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)+'[sim]',
                'torch==2.11.0+cu128', 'torchvision==0.26.0+cu128',
                '--extra-index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
sys.path.insert(0, str(package_dir))
from pnp import env_setup
env_setup.setup_environment()
import torch
if torch.__version__ != '2.11.0+cu128':
    raise RuntimeError('Restart the runtime to load the installed Torch 2.11.0+cu128.')
print({'source_revision': SOURCE_REVISION, 'gpu': torch.cuda.get_device_name(0), 'source': str(package_dir)})


In [ ]:
# Record the actual installed simulator/grader code without changing it.
import inspect, hashlib, json, importlib.metadata
from pathlib import Path
from libero.libero.envs import OffScreenRenderEnv
from pnp.store import SupabaseStore
import libero.libero.envs as libero_envs
EXPERIMENT = 'smolvla-libero-jeff-original-notebook-v1'
root = Path(libero_envs.__file__).parent
manifest = {'source_revision': SOURCE_REVISION,
            'versions': {name:importlib.metadata.version(name) for name in ('libero','mujoco','robosuite','torch','transformers')},
            'check_success_source': inspect.getsource(OffScreenRenderEnv.check_success),
            'libero_env_source_sha256': {str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(root.rglob('*.py'))}}
SupabaseStore()._upload(f'smolvla_pcp_sweep/{EXPERIMENT}/grader_source_manifest.json', json.dumps(manifest,indent=2).encode())
print({'versions':manifest['versions'], 'check_success_source':manifest['check_success_source'],
       'grader_source_manifest_saved':True})


In [ ]:
# Freeze the comparison target and validate the recovered cohort before rollout.
import platform, subprocess, collections
from pnp.experiments import _prepare_libero_episodes
from pnp.rollout import episode_seed
REFERENCE = 'smolvla-libero-a10-pnp-k5-steps34-v1'
store = SupabaseStore()
reference_fields = 'rollout_id,run_id,suite,task_idx,episode_idx,init_state_hash,success,episode_seed,n_steps,bddl_sha256'
reference = store.fetch_all('rollouts', reference_fields,
    configure=lambda q:q.eq('experiment', REFERENCE).eq('method','pnp_uncertainty_only').eq('status','completed'),
    order_by=('rollout_id',))
identity = lambda r:(r['suite'],int(r['task_idx']),int(r.get('episode_idx',r.get('ep_idx'))),r['init_state_hash'])
assert len(reference)==400 and len({identity(r) for r in reference})==400
assert sum(bool(r['success']) for r in reference)==255
old = {identity(r):r for r in reference}
episodes = _prepare_libero_episodes()
assert len(episodes)==400 and {identity(e) for e in episodes}==set(old), 'Episode cohort differs from Jeff'
assert all(e['bddl_sha256']==old[identity(e)]['bddl_sha256'] for e in episodes), 'Task definitions differ'
assert all(episode_seed(e['init_state'],e['ep_idx'])==old[identity(e)]['episode_seed'] for e in episodes), 'Episode seeds differ'
source_runs = store.fetch_all('experiment_runs', '*',
    configure=lambda q:q.in_('run_id',sorted({r['run_id'] for r in reference})),order_by=('created_at','run_id'))
# Do not alter precision to force agreement: verify the recorded settings.
assert torch.cuda.get_device_name(0)=='NVIDIA L4', 'Use L4 to match the recorded GPU'
assert torch.get_float32_matmul_precision()=='highest'
assert not torch.backends.cuda.matmul.allow_tf32
protocol = {
    'reference_experiment':REFERENCE, 'experiment':EXPERIMENT, 'reference_stock_successes':255,
    'planned_identities':400, 'planned_rollouts':800, 'source_revision':SOURCE_REVISION,
    'runtime':{'python':platform.python_version(), 'gpu':torch.cuda.get_device_name(0),
               'driver':subprocess.check_output(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],text=True).strip(),
               'torch':torch.__version__, 'cuda':torch.version.cuda, 'cudnn':torch.backends.cudnn.version(),
               'matmul_precision':torch.get_float32_matmul_precision(),
               'matmul_tf32':torch.backends.cuda.matmul.allow_tf32,
               'cudnn_tf32':torch.backends.cudnn.allow_tf32,
               'cudnn_deterministic':torch.backends.cudnn.deterministic,
               'cudnn_benchmark':torch.backends.cudnn.benchmark,
               'deterministic_algorithms':torch.are_deterministic_algorithms_enabled(),
               'num_threads':torch.get_num_threads()},
    'installed_packages':sorted([(d.metadata.get('Name',''),d.version) for d in importlib.metadata.distributions()]),
    'historical_runs':source_runs,
    'source_files_sha256':{str(f.relative_to(package_dir)):hashlib.sha256(f.read_bytes()).hexdigest() for f in sorted((package_dir/'pnp').rglob('*.py'))},
    'known_differences':['Reference combines seven resumed runs and three source revisions; this fresh run uses final recorded revision throughout.',
                         'Reference retains 12 early batch-size-two episodes; this fresh run uses the original final notebook batch limit eight.',
                         'Resume-dependent batch memberships are not reconstructed by the original notebook.'],
    'unknown_historical_settings':['Exact policy/VLM/tokenizer file hashes','MuJoCo/robosuite versions','GPU driver and all cuDNN settings','Overwritten or interrupted attempts'],
    'interpretation':'A best-effort protocol replication. Outcome agreement is descriptive, not a guarantee of numerical identity. Do not tune settings until the historical total is reproduced.'}
store._upload(f'smolvla_pcp_sweep/{EXPERIMENT}/protocol.json',json.dumps(protocol,indent=2,default=str).encode())
store._upload(f'smolvla_pcp_sweep/{EXPERIMENT}/reference_stock_rows.json',json.dumps(reference,indent=2).encode())
print({'cohort_verified':400,'task_hash_mismatches':0,'seed_mismatches':0,
       'historical_runs':len(source_runs),'runtime':protocol['runtime'],'known_differences':protocol['known_differences']})


In [ ]:
from pnp.experiments import run_smolvla_libero_worker
EXPERIMENT = 'smolvla-libero-jeff-original-notebook-v1'
# Same original runner and arguments as notebook 83; sequential shards fit one Colab runtime.
for shard_index in (0, 1):
    print(f'Running Jeff original notebook shard {shard_index + 1}/2', flush=True)
    run_smolvla_libero_worker(shard_count=2, shard_index=shard_index,
                             experiment=EXPERIMENT, rollout_batch_size=8)


In [ ]:
# Report the comparison without treating matching totals as exact replication.
import math, numpy as np
fields = 'rollout_id,run_id,suite,task_idx,episode_idx,init_state_hash,method,status,success,episode_seed,n_steps,bddl_sha256'
rows = store.fetch_all('rollouts',fields,configure=lambda q:q.eq('experiment',EXPERIMENT),order_by=('rollout_id',))
arm_results = {}
for method in sorted({r['method'] for r in rows}):
    completed = [r for r in rows if r['method']==method and r['status']=='completed']
    count=len(completed); successes=sum(bool(r['success']) for r in completed)
    arm_results[method]={'completed':count,'planned':400,'successes':successes,
                         'errors':sum(r['method']==method and r['status']!='completed' for r in rows)}
    print(method,arm_results[method])
old = {identity(r):r for r in reference}
new = {identity(r):r for r in rows if r['method']=='pnp_uncertainty_only' and r['status']=='completed'}
pairs = [(old[k],new[k]) for k in sorted(old.keys() & new.keys())]
rescues=sum(not a['success'] and b['success'] for a,b in pairs)
spoils=sum(a['success'] and not b['success'] for a,b in pairs)
deltas=np.array([float(b['success'])-float(a['success']) for a,b in pairs])
se=float(deltas.std(ddof=1)/math.sqrt(len(deltas))) if len(deltas)>1 else None
task_groups=collections.defaultdict(list)
for a,b in pairs: task_groups[(a['suite'],a['task_idx'])].append(float(b['success'])-float(a['success']))
task_means=np.array([np.mean(v) for v in task_groups.values()])
task_se=float(task_means.std(ddof=1)/math.sqrt(len(task_means))) if len(task_means)>1 else None
run_sha={r['run_id']:r['pnp_git_sha'] for r in source_runs}
subgroups={}
for sha in sorted(set(run_sha.values())):
    group=[(a,b) for a,b in pairs if run_sha[a['run_id']]==sha]
    subgroups[sha]={'paired':len(group),'historical_successes':sum(a['success'] for a,b in group),
                    'new_successes':sum(b['success'] for a,b in group),
                    'outcome_matches':sum(a['success']==b['success'] for a,b in group)}
report={'reference_experiment':REFERENCE,'replication_experiment':EXPERIMENT,'arm_results':arm_results,
        'paired':len(pairs),'historical_successes':sum(a['success'] for a,b in pairs),
        'new_successes':sum(b['success'] for a,b in pairs),'rescues':rescues,'spoils':spoils,
        'outcome_matches':sum(a['success']==b['success'] for a,b in pairs),
        'seed_mismatches':sum(a['episode_seed']!=b['episode_seed'] for a,b in pairs),
        'step_count_mismatches':sum(a['n_steps']!=b['n_steps'] for a,b in pairs),
        'task_definition_mismatches':sum(a['bddl_sha256']!=b['bddl_sha256'] for a,b in pairs),
        'paired_difference':float(deltas.mean()) if len(deltas) else None,
        'episode_assuming_independence_se':se,'task_group_count':len(task_means),
        'equal_task_weight_difference':float(task_means.mean()) if len(task_means) else None,
        'equal_task_weight_se':task_se,
        'task_uncertainty_note':'SE across task means; interpret the partial-cohort task weighting separately until all 400 finish.',
        'by_historical_source_revision':subgroups,
        'outcome_flips':[{'identity':identity(a),'historical_success':a['success'],'new_success':b['success']} for a,b in pairs if a['success']!=b['success']]}
store._upload(f'smolvla_pcp_sweep/{EXPERIMENT}/comparison.json',json.dumps(report,indent=2).encode())
# Fingerprint the cached main snapshots used by the unmodified original loader.
weight_manifest={}
hub=Path(os.environ['HF_HOME'])/'hub'
for repo in ('HuggingFaceVLA/smolvla_libero','HuggingFaceTB/SmolVLM2-500M-Instruct'):
    cache=hub/('models--'+repo.replace('/','--')); ref=cache/'refs'/'main'
    if ref.exists():
        revision=ref.read_text().strip(); snapshot=cache/'snapshots'/revision
        weight_manifest[repo]={'revision':revision,'files_sha256':{str(f.relative_to(snapshot)):hashlib.sha256(f.read_bytes()).hexdigest() for f in sorted(snapshot.rglob('*')) if f.is_file()}}
store._upload(f'smolvla_pcp_sweep/{EXPERIMENT}/model_files_manifest.json',json.dumps(weight_manifest,indent=2).encode())
print(json.dumps({k:v for k,v in report.items() if k!='outcome_flips'},indent=2))
